# Notebook 04: Weather and Terrain Feature Engineering
**GlobalFlood AI: Global Multimodal Flood Prediction System (48-Hour Forecast)**

### Objective
Engineer atmospheric and topographic features with strict leakage prevention:
1. **168-Hour Historical Weather Sequence:** Captures antecedent soil saturation using ERA5 reanalysis
2. **48-Hour Forecast Weather Sequence:** Captures future precipitation and atmospheric forcing
3. **Missingness Masking:** Produces paired binary masks indicating which hours and variables are observed
4. **Topographic Features:** Elevation, slope, Terrain Roughness Index (TRI), Topographic Wetness Index (TWI)

### Inputs & Outputs
- **Inputs:** Coordinates $(lat, lon)$ and prediction timestamp $T_0$
- **Outputs:** Normalized arrays of shapes $(168, 6)$ and $(48, 6)$ with paired masks $(168, 6)$ and $(48, 6)$, and 9-d terrain vector
- **Dependencies:** `httpx`, `numpy`, `ml.src.preprocessing`


In [ ]:
import httpx
from ml.src.preprocessing import MultimodalPreprocessor

lat, lon = 13.0827, 80.2707

# Fetch weather using Open-Meteo
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": lat,
    "longitude": lon,
    "hourly": ["precipitation", "temperature_2m", "relative_humidity_2m", "surface_pressure", "wind_speed_10m", "soil_moisture_0_to_1cm"],
    "past_days": 7,
    "forecast_days": 2,
    "timezone": "UTC"
}
with httpx.Client(timeout=15.0) as client:
    resp = client.get(url, params=params)
    data = resp.json()

hourly = data["hourly"]
times = hourly["time"]
print(f"Total hourly timestamps returned: {len(times)}")

# Partition into 168h historical observations vs 48h future forecast
hist_records = []
for i in range(min(168, len(times))):
    hist_records.append({
        "precipitation_mm": hourly["precipitation"][i],
        "temperature_2m_c": hourly["temperature_2m"][i],
        "relative_humidity_pct": hourly["relative_humidity_2m"][i],
        "surface_pressure_hpa": hourly["surface_pressure"][i],
        "wind_speed_10m_kmh": hourly["wind_speed_10m"][i],
        "soil_moisture_m3m3": hourly["soil_moisture_0_to_1cm"][i] if "soil_moisture_0_to_1cm" in hourly else None
    })

fc_records = []
for i in range(168, min(168 + 48, len(times))):
    fc_records.append({
        "precipitation_mm": hourly["precipitation"][i],
        "temperature_2m_c": hourly["temperature_2m"][i],
        "relative_humidity_pct": hourly["relative_humidity_2m"][i],
        "surface_pressure_hpa": hourly["surface_pressure"][i],
        "wind_speed_10m_kmh": hourly["wind_speed_10m"][i],
        "soil_moisture_m3m3": None
    })

print(f"Extracted {len(hist_records)} historical hours and {len(fc_records)} forecast hours.")


### Preprocessing Weather Sequences
Transforms raw hourly dictionaries into standardized feature matrices and missingness masks.


In [ ]:
hist_vals, hist_mask = MultimodalPreprocessor.preprocess_weather_sequence(hist_records, target_hours=168)
fc_vals, fc_mask = MultimodalPreprocessor.preprocess_weather_sequence(fc_records, target_hours=48)

print("Historical weather tensor shape:", hist_vals.shape, "Mask shape:", hist_mask.shape)
print("Forecast weather tensor shape:  ", fc_vals.shape, "Mask shape:", fc_mask.shape)
assert hist_vals.shape == (168, 6)
assert fc_vals.shape == (48, 6)
assert hist_mask.shape == (168, 6)
assert fc_mask.shape == (48, 6)


### Topographic Feature Engineering
Combines elevation, slope, TRI, TWI, and spherical coordinate encoding $[\sin(lat), \cos(lat), \sin(lon), \cos(lon)]$.


In [ ]:
terrain_dict = {
    "elevation_m": 12.0,
    "slope_degrees": 1.1,
    "terrain_roughness_index": 2.2,
    "topographic_wetness_index": 11.0,
    "estimated_dist_to_drainage_km": 1.2
}
geo_vec, geo_avail = MultimodalPreprocessor.preprocess_geo_features(lat, lon, terrain_dict)
print("Geographic feature vector shape:", geo_vec.shape, f"Availability: {geo_avail}")
print("Feature vector:", np.round(geo_vec, 3))
assert geo_vec.shape == (9,)
print("\n[SUCCESS] Weather sequences and topographic features engineered and verified.")
